### **This notebook is old and unused!**

In [ ]:
import os
import json
import pathlib
import asyncio
import time
from typing import List

import pymupdf4llm
from tqdm import tqdm

from pydantic_ai import Agent, StructuredDict
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

# === timing helper ===
def timed_step(name: str, start_time: float) -> float:
    elapsed = time.perf_counter() - start_time
    print(f"[✓] {name} completed in {elapsed:.2f} seconds.")
    return time.perf_counter()

global_start = time.perf_counter()

# === Paths setup ===
benchmark_root = pathlib.Path("./benchmark_papers")
pdf_dir = benchmark_root
md_cache_dir = benchmark_root / "markdown_cache"
json_root_dir = pathlib.Path("./benchmark_output")

md_cache_dir.mkdir(parents=True, exist_ok=True)
json_root_dir.mkdir(parents=True, exist_ok=True)

# Safety check (optional but handy)
if not pdf_dir.exists():
    raise FileNotFoundError(f"PDF directory not found: {pdf_dir.resolve()}")

# === Load system prompt and schema ===
context_path = pathlib.Path("./helpers/context.md")
with context_path.open("r", encoding="utf-8") as f:
    system_prompt_text = f.read()

with open("./helpers/WeakSchema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

ProjectPublications = StructuredDict(
    schema,
    name="ProjectPublications",
    description="Extracted project publications data."
)

# === Initialize OpenRouter provider ===
openrouter_api_key = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
provider = OpenRouterProvider(api_key=openrouter_api_key)

# === List of models to benchmark ===
MODEL_NAMES = [
    "openai/gpt-5-mini",
    "meta-llama/llama-3.3-70b-instruct:nitro",
    "openai/gpt-5"
]

def safe_model_dir_name(model_name: str) -> str:
    """
    Convert a model name into a filesystem-safe folder name.
    Keeps it readable while avoiding '/' and ':' issues.
    """
    return (
        model_name
        .replace("/", "__")
        .replace(":", "__")
        .replace(" ", "_")
    )

def create_agent_for_model(model_name: str) -> Agent:
    """
    Create a PydanticAI Agent for a given model with the shared system prompt
    and ProjectPublications structured output.
    """
    model = OpenAIChatModel(model_name, provider=provider)
    agent = Agent(model, output_type=ProjectPublications)

    @agent.system_prompt
    def _system_prompt() -> str:
        return system_prompt_text

    return agent


In [ ]:
async def process_single_pdf(pdf_path: pathlib.Path, agent: Agent) -> dict:
    """
    Convert a single PDF to Markdown (with cache), run the agent, and return the output dict.
    """
    pdf_stem = pdf_path.stem
    cached_md_path = md_cache_dir / f"{pdf_stem}.md"

    # Load or create Markdown cache
    if cached_md_path.exists():
        md_text = cached_md_path.read_text(encoding="utf-8")
    else:
        md_text = pymupdf4llm.to_markdown(str(pdf_path))
        cached_md_path.write_text(md_text, encoding="utf-8")

    # Run agent for structured extraction
    response = await agent.run(
        md_text,
        model_settings={"temperature": 0},
    )

    return response.output


async def process_all_pdfs_for_model(
    pdf_paths: List[pathlib.Path],
    agent: Agent,
    model_output_dir: pathlib.Path,
):
    """
    Process all PDFs for a single model, save JSONs into model_output_dir,
    and print per-file + summary stats.
    """
    results = []

    for pdf_path in tqdm(pdf_paths, desc=f"Processing PDFs → JSON ({model_output_dir.name})"):
        try:
            data = await process_single_pdf(pdf_path, agent)

            stem = pdf_path.stem
            json_path = model_output_dir / f"{stem}.json"

            json_str = json.dumps(data, indent=2, ensure_ascii=False)
            json_path.write_text(json_str, encoding="utf-8")

            # Collect stats
            schema_props = schema.get("properties", {})
            total_params = len(schema_props)
            missing_fields = [k for k in schema_props if k not in data or data[k] is None]
            missing_count = len(missing_fields)

            results.append({
                "pdf": str(pdf_path),
                "json": str(json_path),
                "total_params": total_params,
                "missing_count": missing_count,
                "missing_fields": missing_fields,
            })

            print(f"\n[📊] {stem}: {missing_count}/{total_params} fields missing")

        except Exception as e:
            print(f"[!] Error processing {pdf_path}: {e}")
            continue

    # === Summary for this model ===
    if not results:
        print(f"\n[!] No successful results for model directory {model_output_dir}")
        return

    avg_missing = sum(item["missing_count"] for item in results) / len(results)
    total_processed = len(results)

    print(f"\n=== SUMMARY for {model_output_dir.name} ===")
    print(f"Total PDFs processed: {total_processed}")
    print(f"Average missing fields per file: {avg_missing:.1f}")
    print(f"JSON outputs saved in: {model_output_dir}")

    # Print sample output for first successful file
    first_result = results[0]
    print(f"\n=== SAMPLE OUTPUT ({first_result['pdf']}) ===")
    with open(first_result["json"], "r", encoding="utf-8") as f:
        sample = json.load(f)
    print(json.dumps(sample, indent=2)[:500] + "...")


async def run_benchmark_for_all_models():
    pdf_paths = sorted(pdf_dir.glob("*.pdf"))
    if not pdf_paths:
        print(f"[!] No PDFs found in {pdf_dir.resolve()}.")
        return

    for model_name in MODEL_NAMES:
        print(f"\n\n##############################")
        print(f"### Running model: {model_name}")
        print(f"##############################\n")

        agent = create_agent_for_model(model_name)
        model_dir_name = safe_model_dir_name(model_name)
        model_output_dir = json_root_dir / model_dir_name
        model_output_dir.mkdir(parents=True, exist_ok=True)

        t_model_start = time.perf_counter()
        await process_all_pdfs_for_model(pdf_paths, agent, model_output_dir)
        timed_step(f"Completed processing for {model_name}", t_model_start)

    total_runtime = time.perf_counter() - global_start
    print(f"\n=== TOTAL PIPELINE RUNTIME (all models): {total_runtime:.2f} seconds ===")


# === Run everything (Jupyter-friendly) ===
await run_benchmark_for_all_models()
